# Module 5 — Orchestrating Continuous Data Pipelines with Snowflake

You built the transformation logic in Module 3. Now you'll wrap it in an automated pipeline: a **Task** that runs the stream-processing sproc on a schedule, plus a downstream Task that sends a weekly report email — a small **DAG** entirely inside Snowflake.

## Learning objectives
- Schedule a Snowpark stored procedure with a **Task**.
- Chain Tasks into a **DAG** using the `AFTER` clause.
- Send transactional emails from Snowflake using a **Notification Integration**.
- Inspect Task run history for observability.

## Prerequisites
- Module 3 completed. `tasty_bytes.raw_pos.process_order_headers_stream` procedure and `order_header_stream` stream exist.
- A valid email address you want the weekly report sent to.

## What you'll build
- Task `process_orders_header_sproc` — calls the sproc every minute.
- Notification Integration `email_notification_int` — enables SYSTEM$SEND_EMAIL.
- Sproc `last_seven_days_report` — queries the last 7 days of Hamburg sales and renders them as an HTML email.
- Task `send_last_seven_days_report` — chained after the first Task, sends the email.
- One end-to-end DAG execution.

## 1. Set your session context

In [ ]:
USE ROLE accountadmin;
USE WAREHOUSE compute_wh;
USE DATABASE tasty_bytes;

## 2. Schedule the sproc with a Task

The Task below runs `process_order_headers_stream()` every minute using a standard CRON schedule. Tasks are created in a suspended state, so we explicitly `RESUME` it.

In [ ]:
-- Task that runs executes every minute
CREATE OR REPLACE TASK tasty_bytes.raw_pos.process_orders_header_sproc
WAREHOUSE = 'COMPUTE_WH'
SCHEDULE = 'USING CRON * * * * * UTC'
AS
CALL tasty_bytes.raw_pos.process_order_headers_stream();

In [ ]:
-- Activate the task to run
ALTER TASK tasty_bytes.raw_pos.process_orders_header_sproc RESUME;

Peek at the destination table and the Task run history to confirm things are moving:

In [ ]:
SELECT * FROM tasty_bytes.raw_pos.daily_sales_hamburg_t;

In [ ]:
-- See run history for task
SELECT *
FROM TABLE(INFORMATION_SCHEMA.TASK_HISTORY())
WHERE NAME = 'PROCESS_ORDERS_HEADER_SPROC'
ORDER BY SCHEDULED_TIME DESC
LIMIT 10;

Insert another Hamburg order. Within 1 minute the Task should pick it up, run the sproc, and land the row in `daily_sales_hamburg_t`.

In [ ]:
-- Insert some dummy data into ORDER_HEADER
INSERT INTO tasty_bytes.raw_pos.order_header (
    ORDER_ID, 
    TRUCK_ID, 
    LOCATION_ID, 
    CUSTOMER_ID, 
    DISCOUNT_ID, 
    SHIFT_ID, 
    SHIFT_START_TIME, 
    SHIFT_END_TIME, 
    ORDER_CHANNEL, 
    ORDER_TS, 
    SERVED_TS, 
    ORDER_CURRENCY, 
    ORDER_AMOUNT, 
    ORDER_TAX_AMOUNT, 
    ORDER_DISCOUNT_AMOUNT, 
    ORDER_TOTAL
) VALUES (
    123456789, 101, 4494, null, null, 123456789,
    '08:00:00', '16:00:00', null,
    '2024-01-12 12:30:45', null, 'USD',
    22.00, null, null, 24.50
);

Wait a minute, then re-query the destination table:

In [ ]:
SELECT * FROM tasty_bytes.raw_pos.daily_sales_hamburg_t;

Use a Streamlit table to eyeball the running Task list in one glance:

In [ ]:
import streamlit as st
from snowflake.snowpark.context import get_active_session

session = get_active_session()

tasks_df = session.sql("SHOW TASKS IN SCHEMA tasty_bytes.raw_pos").to_pandas()
st.subheader("Tasks in tasty_bytes.raw_pos")
st.dataframe(tasks_df[["name", "state", "schedule", "predecessors"]], use_container_width=True)

Suspend the Task while we build out the DAG so it doesn't fire mid-setup:

In [ ]:
ALTER TASK tasty_bytes.raw_pos.process_orders_header_sproc SUSPEND;

## 3. Add a downstream email report to build a DAG

We'll extend the pipeline with a second Task that runs **after** the first one and emails a weekly Hamburg sales summary. Three pieces:
1. A **Notification Integration** that authorizes email delivery.
2. A **Snowpark Python sproc** that queries the last 7 days of Hamburg sales and formats an HTML email.
3. A **child Task** that runs `AFTER` the first Task and calls the sproc.

**Update the email address below** to your own before running the next cell.

In [ ]:
-- Create an email integration 
CREATE OR REPLACE NOTIFICATION INTEGRATION email_notification_int
TYPE = EMAIL
ENABLED = TRUE
ALLOWED_RECIPIENTS = ('email@address.com');  -- Update the recipient's email here

### The weekly report sproc

This stored procedure pulls the 7 most recent rows from `daily_sales_hamburg_t`, converts them into a styled HTML table, and calls `SYSTEM$SEND_EMAIL` through the integration. Update `email@address.com` in two places (recipient + email body) to match your address.

In [ ]:
CREATE OR REPLACE PROCEDURE tasty_bytes.raw_pos.last_seven_days_report()
RETURNS STRING
LANGUAGE PYTHON
RUNTIME_VERSION = '3.10'
PACKAGES = ('snowflake-snowpark-python')
HANDLER = 'send_email'
AS 
$$
import snowflake.snowpark.functions as F
from snowflake.snowpark import Session

def send_email(session: Session) -> str:
    # Query the 7 most recent entries in the DAILY_SALES_HAMBURG_T table
    recent_entries_df = session.table("RAW_POS.DAILY_SALES_HAMBURG_T") \
                               .sort(F.col("DATE_TRUNC(LITERAL(), ORDER_TS)").desc()) \
                               .limit(7) \
                               .to_pandas()
    
    # Convert the DataFrame to an HTML table with styling
    html_table = recent_entries_df.to_html(index=False, classes='styled-table')

    # Define the email content with Snowflake branding and custom styling
    email_content = f"""
    <html>
    <head>
        <style>
            body {{
                font-family: Arial, sans-serif;
            }}
            h2 {{
                color: #29B5E8;
            }}
            .styled-table {{
                border-collapse: collapse;
                margin: 25px 0;
                font-size: 0.9em;
                font-family: 'Trebuchet MS', 'Lucida Sans Unicode', 'Lucida Grande', 'Lucida Sans', Arial, sans-serif;
                min-width: 400px;
                border-radius: 5px 5px 0 0;
                overflow: hidden;
                box-shadow: 0 0 20px rgba(0, 0, 0, 0.15);
            }}
            .styled-table thead tr {{
                background-color: #0072CE;
                color: #ffffff;
                text-align: left;
                font-weight: bold;
            }}
            .styled-table th,
            .styled-table td {{
                padding: 12px 15px;
            }}
            .styled-table tbody tr {{
                border-bottom: 1px solid #dddddd;
            }}
            .styled-table tbody tr:nth-of-type(even) {{
                background-color: #f3f3f3;
            }}
            .styled-table tbody tr:last-of-type {{
                border-bottom: 2px solid #0072CE;
            }}
        </style>
    </head>
    <body>
        <h2>Weekly Sales Report for Hamburg</h2>
        <p>Here are the last 7 entries in the <strong>DAILY_SALES_HAMBURG_T</strong> table:</p>
        {html_table}
    </body>
    </html>
    """
    
    # Send the email
    session.call("system$send_email",
                 "email_notification_int",
                 "email@address.com",
                 "Weekly Sales Report for Hamburg",
                 email_content,
                 "text/html")
    
    # Return a success message
    return "Weekly sales report email sent successfully"
$$;

### Chain the child Task after the parent

`AFTER tasty_bytes.raw_pos.process_orders_header_sproc` establishes the DAG edge. The child Task inherits its schedule from the root; it fires only after the root completes.

In [ ]:
-- Create task that runs after the process_orders_header_sproc task
-- It calls the sproc above
CREATE OR REPLACE TASK tasty_bytes.raw_pos.send_last_seven_days_report
WAREHOUSE = 'COMPUTE_WH'
AFTER tasty_bytes.raw_pos.process_orders_header_sproc
AS
CALL tasty_bytes.raw_pos.last_seven_days_report();

Resume both Tasks and kick off a manual DAG run with `EXECUTE TASK` on the root — the child fires automatically after the root succeeds.

In [ ]:
-- Start the tasks. Child MUST be resumed BEFORE the root, otherwise the DAG edge won't wire up.
ALTER TASK tasty_bytes.raw_pos.send_last_seven_days_report RESUME;
ALTER TASK tasty_bytes.raw_pos.process_orders_header_sproc RESUME;

-- Kick off the DAG immediately (don't wait for the next CRON tick)
EXECUTE TASK tasty_bytes.raw_pos.process_orders_header_sproc;

Watch the DAG execute. `COMPLETE_TASK_GRAPHS` shows finished graph runs; `TASK_HISTORY` shows individual Task runs including the child.

In [ ]:
SELECT *
FROM TABLE(INFORMATION_SCHEMA.COMPLETE_TASK_GRAPHS(
  ROOT_TASK_NAME => 'PROCESS_ORDERS_HEADER_SPROC'
))
ORDER BY SCHEDULED_TIME DESC
LIMIT 5;

In [ ]:
SELECT *
FROM TABLE(INFORMATION_SCHEMA.TASK_HISTORY())
WHERE NAME IN ('PROCESS_ORDERS_HEADER_SPROC', 'SEND_LAST_SEVEN_DAYS_REPORT')
ORDER BY SCHEDULED_TIME DESC
LIMIT 10;

## 4. Teardown

Suspend both Tasks before you close out — otherwise they will keep running (and billing) every minute forever.

In [ ]:
ALTER TASK tasty_bytes.raw_pos.process_orders_header_sproc SUSPEND;
ALTER TASK tasty_bytes.raw_pos.send_last_seven_days_report SUSPEND;

### Optional: drop everything you created

Uncomment and run to remove all Task/sproc/integration objects created in this module.

In [ ]:
-- DROP TASK IF EXISTS tasty_bytes.raw_pos.send_last_seven_days_report;
-- DROP TASK IF EXISTS tasty_bytes.raw_pos.process_orders_header_sproc;
-- DROP PROCEDURE IF EXISTS tasty_bytes.raw_pos.last_seven_days_report();
-- DROP NOTIFICATION INTEGRATION IF EXISTS email_notification_int;

## Recap

You wired the Module 3 transformation into a continuously running pipeline: **Stream → Sproc → Task → Child Task → Email**. Everything runs inside Snowflake with no external orchestrator, no external notification service, and no data movement.

That closes the loop for this course — from raw ingestion (Module 2) → transformation (Module 3) → delivery (Module 4) → automation (Module 5).